In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import cm
import umap
import re
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
from tqdm import tqdm

pos_tsv_path = r"C:\Users\ankit\Downloads\Lab Test\positive_explanations_70_test.tsv"
pos_npz_path = r"C:\Users\ankit\Downloads\Lab Test\positive_explanations_70_test.npz"
output_pdf = r"C:\Users\ankit\Downloads\Lab Test\kmeans_cosine_charged_only_70_test.pdf"
descriptor_csv = r"C:\Users\ankit\Downloads\Lab Test\cluster_sequence_descriptors_70_test.csv"
anarci_aligned_path = r"D:\Thesis EGFR Round 3\Dataset\ANARCII\final_EG-ANARCII_imgt_wide_fixed_filtered.csv"
anarci_sep = ","  
top_fraction = 0.20
aggregation = "mean"   
n_examples_per_cluster = 20
random_state = 42
min_k = 5
max_k = 40
umap_n_neighbors = 15
umap_min_dist = 0.1
aa_labels = sorted(list("ACDEFGHIKLMNPQRSTVWY")) + ['-']
canonical_aas = sorted(list("ACDEFGHIKLMNPQRSTVWY"))
aa_to_index_20 = {aa: i for i, aa in enumerate(canonical_aas)}
aa_to_index_full = {aa: i for i, aa in enumerate(aa_labels)}
charged_aas = ["D", "E", "H", "K", "R"]
charged_channel_indices = [aa_to_index_20[aa] for aa in charged_aas]
sequence_fontsize = 9
seq_left_margin = 0.6
heatmap_cmap = "viridis"
cmap = plt.get_cmap(heatmap_cmap).copy()
cmap.set_bad(color="white")
cluster_map_cmap_name = "tab20b"
overlap_cmap_name = "Blues"

def build_residue_position_tensor(sequence, mat, aa_to_index_20, aa_to_index_full):
    sequence = str(sequence)
    L = len(sequence)
    if mat.ndim != 2:
        raise ValueError(f"Expected 2D matrix, got shape {mat.shape}")
    if mat.shape[0] != L:
        raise ValueError(
            f"Sequence length ({L}) does not match matrix positions ({mat.shape[0]}). "
            f"Sequence: {sequence}")
    if mat.shape[1] != len(aa_labels):
        raise ValueError(
            f"Expected matrix with {len(aa_labels)} amino acid columns, got {mat.shape[1]}")
    tensor = np.zeros((20, L), dtype=float)
    for pos, aa in enumerate(sequence):
        if aa not in aa_to_index_20:
            continue
        if aa not in aa_to_index_full:
            continue
        channel_idx = aa_to_index_20[aa]
        col_idx = aa_to_index_full[aa]
        tensor[channel_idx, pos] = float(mat[pos, col_idx])
    return tensor

def build_position_overlap_strength(sequences, min_overlap=5):
    if not sequences:
        return np.array([])
    lengths = {len(seq) for seq in sequences}
    if len(lengths) != 1:
        raise ValueError(f"All selected sequences must have the same length, got lengths={sorted(lengths)}")
    seq_arr = np.array([list(seq) for seq in sequences])
    overlap = np.zeros(seq_arr.shape[1], dtype=float)
    for pos in range(seq_arr.shape[1]):
        vals, counts = np.unique(seq_arr[:, pos], return_counts=True)
        max_count = counts.max()
        overlap[pos] = max_count if max_count >= min_overlap else np.nan
    return overlap

def draw_sequence_stack(ax, sequences, pred_probs, rounds, overlap_strength, title):
    n_seq = len(sequences)
    if n_seq == 0:
        ax.axis("off")
        return
    seq_len = len(sequences[0])
    color_img = np.ones((n_seq, seq_len, 3), dtype=float)
    for x in range(seq_len):
        overlap_val = overlap_strength[x] if x < len(overlap_strength) else np.nan
        if not np.isnan(overlap_val) and overlap_val >= 5:
            intensity = (min(overlap_val, 10) - 5) / 5.0
            green_rgb = np.array([0.85 - 0.45 * intensity, 1.00, 0.85 - 0.45 * intensity])
            color_img[:, x, :] = green_rgb
    for row_idx, seq in enumerate(sequences):
        for x, aa in enumerate(seq):
            if aa in {"R", "K", "H"}:
                if aa == "R":
                    intensity = 1.0
                elif aa == "K":
                    intensity = 0.8
                else:
                    intensity = 0.5
                red_rgb = np.array([
                    1.00, 0.85 - 0.70 * intensity, 0.85 - 0.70 * intensity])
                color_img[row_idx, x, :] = red_rgb
            elif aa in {"D", "E"}:
                if aa == "D":
                    intensity = 0.8
                else:
                    intensity = 1.0
                blue_rgb = np.array([0.85 - 0.70 * intensity, 0.85 - 0.70 * intensity, 1.00])
                color_img[row_idx, x, :] = blue_rgb
    ax.imshow(color_img, aspect="auto", interpolation="nearest", extent=(-0.5, seq_len - 0.5, n_seq - 0.5, -0.5))
    for row_idx, (seq, pred_prob) in enumerate(zip(sequences, pred_probs)):
        y = row_idx
        ax.text(seq_left_margin, y, f"{row_idx + 1:02d}  R{rounds[row_idx]}  p={pred_prob:.3f}", ha="right", va="center", fontsize=sequence_fontsize, family="monospace")
        for x, aa in enumerate(seq):
            ax.text(x, y, aa, ha="center", va="center", fontsize=sequence_fontsize, family="monospace")
    ax.set_xlim(-1.8, seq_len - 0.5)
    ax.set_ylim(n_seq - 0.5, -0.5)
    ax.set_xticks(np.arange(seq_len))
    ax.set_xticklabels(anarci_positions, fontsize=8, rotation=90)
    ax.set_xlabel("ANARCII / IMGT position")
    ax.set_yticks([])
    ax.set_title(title, fontsize=11)
    for spine in ax.spines.values():
        spine.set_visible(False)

def select_top_nonredundant_cluster_examples(cluster_df, n_examples=10):
    return (cluster_df.sort_values("Pred_prob_1", ascending=False).drop_duplicates(subset=["CDRs"], keep="first").head(n_examples).copy())

pos_df = pd.read_csv(pos_tsv_path, sep="\t")
pos_npz = np.load(pos_npz_path, allow_pickle=True)
required_cols = {"True_label", "Pred_label", "Pred_prob_1", "npz_key", "Nanobody_id", "CDRs", "Round"}
missing = required_cols - set(pos_df.columns)
if missing:
    raise ValueError(f"Missing required columns in TSV: {sorted(missing)}")
correct_pos = pos_df[(pos_df["True_label"] == 1) & (pos_df["Pred_label"] == 1)].copy()
if len(correct_pos) == 0:
    raise ValueError("No correctly predicted positive samples found.")
n_top = max(1, int(math.ceil(len(correct_pos) * top_fraction)))
top_pos = correct_pos.sort_values("Pred_prob_1", ascending=False).head(n_top).copy()
top_pos = top_pos.reset_index(drop=True)
print(f"Selected top positives: {len(top_pos)}")
anarci_df = pd.read_csv(anarci_aligned_path, sep=anarci_sep)
all_anarci_cols = [
c for c in anarci_df.columns
    if str(c)[0].isdigit()]

cdr_cols = []
for c in all_anarci_cols:
    num = int("".join([x for x in str(c) if x.isdigit()]))
    if (27 <= num <= 38 or 56 <= num <= 65 or 105 <= num <= 117):
        cdr_cols.append(c)
anarci_positions = [str(c) for c in cdr_cols]
print("First positions:", anarci_positions[:10])
print("Last positions:", anarci_positions[-10:])
print("Total positions:", len(anarci_positions))

mats = []
ids = []
sequences = []
for _, row in top_pos.iterrows():
    key = row["npz_key"]
    if key not in pos_npz:
        raise KeyError(f"npz_key '{key}' not found in NPZ.")
    mat = np.array(pos_npz[key], dtype=float)
    seq = str(row["CDRs"])
    mats.append(mat)
    ids.append(row["Nanobody_id"])
    sequences.append(seq)

shapes = {m.shape for m in mats}
if len(shapes) != 1:
    raise ValueError(f"Not all explanation matrices have the same shape: {shapes}")
matrix_shape = mats[0].shape
print("Matrix shape:", matrix_shape)
if len(anarci_positions) != matrix_shape[0]:
    raise ValueError(
        f"Mismatch:\n"
        f"ANARCII positions = {len(anarci_positions)}\n"
        f"Matrix rows = {matrix_shape[0]}\n"
        f"Sequence length = {len(sequences[0])}\n"
        "Fix column selection.")
print("ANARCII mapping is correct")
if matrix_shape[1] != len(aa_labels):
    raise ValueError(
        f"Expected explanation matrices with {len(aa_labels)} columns "
        f"(20 amino acids + '-'), got shape {matrix_shape}")

seq_lengths = {len(s) for s in sequences}
if len(seq_lengths) != 1:
    raise ValueError(f"Not all CDR sequences have the same length: {sorted(seq_lengths)}")
if list(seq_lengths)[0] != matrix_shape[0]:
    raise ValueError(
        f"Sequence length ({list(seq_lengths)[0]}) does not match matrix row count ({matrix_shape[0]}).")

print("Sequence length:", len(sequences[0]))
print("Matrix rows:", matrix_shape[0])
print("ANARCII CDR labels:", len(anarci_positions))

assert len(anarci_positions) == len(sequences[0]) == matrix_shape[0]
raw_residue_tensors = []
feature_vectors = []
for seq, mat in tqdm(list(zip(sequences, mats)), total=len(sequences), desc="Building charged-only residue maps"):
    residue_tensor = build_residue_position_tensor(
        sequence=seq,
        mat=mat,
        aa_to_index_20=aa_to_index_20,
        aa_to_index_full=aa_to_index_full,)
    raw_residue_tensors.append(residue_tensor)
    charged_only_tensor = residue_tensor[charged_channel_indices, :]
    feature_vectors.append(charged_only_tensor.flatten())

X_features = np.stack(feature_vectors, axis=0)
print("Charged-only feature matrix shape:", X_features.shape)
if X_features.shape[0] < 2:
    raise ValueError("Need at least 2 samples for clustering.")
X_cluster = normalize(X_features, norm="l2", axis=1)
print("Normalized clustering matrix shape:", X_cluster.shape)

n_samples = X_cluster.shape[0]
effective_min_k = max(2, min_k)
effective_max_k = min(max_k, n_samples - 1)
if effective_min_k > effective_max_k:
    raise ValueError(
        f"Not enough samples ({n_samples}) for requested cluster search range {min_k}-{max_k}. "
        f"Valid max k is {n_samples - 1}.")
k_values = list(range(effective_min_k, effective_max_k + 1))
silhouette_scores = []
best_k = None
best_score = -np.inf
best_labels = None
best_model = None

print(f"Searching k in range [{effective_min_k}, {effective_max_k}] using silhouette score")
for k in k_values:
    model = KMeans(n_clusters=k, random_state=random_state, n_init=20)
    labels_k = model.fit_predict(X_cluster)
    unique_k = np.unique(labels_k)
    if len(unique_k) < 2:
        score = -1.0
    else:
        score = silhouette_score(X_cluster, labels_k, metric="euclidean")
    silhouette_scores.append(score)
    print(f"k={k:2d} | silhouette={score:.4f}")
    if score > best_score:
        best_score = score
        best_k = k
        best_labels = labels_k.copy()
        best_model = model
if best_k is None:
    raise ValueError("Could not determine best k from silhouette search.")
labels = best_labels
top_pos["Cluster"] = labels

POSITIVE = set("RKH")
NEGATIVE = set("DE")
AROMATIC = set("FYW")
HYDROPHOBIC = set("AILMFWVPGC")
POLAR = set("STNQCY")
CHARGED = set("RKHDE")

def clean_seq(seq):
    if pd.isna(seq):
        return ""
    return str(seq).replace("-", "").replace(" ", "").upper()

def clean_aligned_seq(seq):
    if pd.isna(seq):
        return ""
    return str(seq).replace(" ", "").upper()

def has_glyco_motif(seq):
    seq = clean_seq(seq)
    return bool(re.search(r"N[^P][ST]", seq))

def max_repeat(seq):
    seq = clean_seq(seq)
    if len(seq) == 0:
        return 0
    return max(len(list(g)) for _, g in __import__("itertools").groupby(seq))

def seq_desc(seq):
    seq = clean_seq(seq)
    L = len(seq)
    if L == 0:
        return pd.Series({
            "length": 0, "net_charge": 0, "positive_count": 0, "negative_count": 0, "charged_count": 0, "aromatic_count": 0, "hydrophobic_count": 0, "polar_count": 0, "cysteine_count": 0,
            "glyco_motif": False, "max_repeat": 0, "charge_density": 0, "aromatic_fraction": 0, "hydrophobic_fraction": 0, "polar_fraction": 0,})

    pos = sum(a in POSITIVE for a in seq)
    neg = sum(a in NEGATIVE for a in seq)
    charged = sum(a in CHARGED for a in seq)
    aromatic = sum(a in AROMATIC for a in seq)
    hydrophobic = sum(a in HYDROPHOBIC for a in seq)
    polar = sum(a in POLAR for a in seq)
    return pd.Series({
        "length": L, "net_charge": pos - neg, "positive_count": pos, "negative_count": neg, "charged_count": charged, "aromatic_count": aromatic, "hydrophobic_count": hydrophobic,
        "polar_count": polar, "cysteine_count": seq.count("C"),"glyco_motif": has_glyco_motif(seq),"max_repeat": max_repeat(seq),"charge_density": (pos - neg) / L, 
        "aromatic_fraction": aromatic / L, "hydrophobic_fraction": hydrophobic / L, "polar_fraction": polar / L,})
    
def flags(seq):
    seq = clean_seq(seq)
    f = []
    if seq.count("C") > 0:
        f.append("Cys present")
    if seq.count("C") > 1:
        f.append("Multiple Cys")
    if has_glyco_motif(seq):
        f.append("N-glycosylation motif")
    if "RRR" in seq or "KKK" in seq:
        f.append("positive repeat")
    if "DDD" in seq or "EEE" in seq:
        f.append("negative repeat")
    if "YYY" in seq or "FFF" in seq or "WWW" in seq:
        f.append("aromatic repeat")
    if max_repeat(seq) >= 4:
        f.append("long repeat")
    return "; ".join(f) if f else "None"

descriptor_df = pd.concat(
    [select_top_nonredundant_cluster_examples(
            top_pos[top_pos["Cluster"] == cid],
            n_examples=n_examples_per_cluster)
        for cid in sorted(top_pos["Cluster"].unique())],ignore_index=True)
print("Selected sequences for descriptor CSV:", len(descriptor_df))

cdr1_indices = []
cdr2_indices = []
cdr3_indices = []
for i, pos in enumerate(anarci_positions):
    num = int("".join([x for x in str(pos) if x.isdigit()]))
    if 27 <= num <= 38:
        cdr1_indices.append(i)
    elif 56 <= num <= 65:
        cdr2_indices.append(i)
    elif 105 <= num <= 117:
        cdr3_indices.append(i)

def extract_region(seq, indices):
    seq = clean_aligned_seq(seq)
    return "".join(seq[i] for i in indices)

descriptor_df["CDR1_aligned"] = descriptor_df["CDRs"].apply(lambda s: extract_region(s, cdr1_indices))
descriptor_df["CDR2_aligned"] = descriptor_df["CDRs"].apply(lambda s: extract_region(s, cdr2_indices))
descriptor_df["CDR3_aligned"] = descriptor_df["CDRs"].apply(lambda s: extract_region(s, cdr3_indices))
descriptor_df["CDR1_seq"] = descriptor_df["CDR1_aligned"].apply(clean_seq)
descriptor_df["CDR2_seq"] = descriptor_df["CDR2_aligned"].apply(clean_seq)
descriptor_df["CDR3_seq"] = descriptor_df["CDR3_aligned"].apply(clean_seq)
descriptor_df["CDR_all_seq"] = (
    descriptor_df["CDR1_seq"] +
    descriptor_df["CDR2_seq"] +
    descriptor_df["CDR3_seq"])

for region in ["CDR1_seq", "CDR2_seq", "CDR3_seq", "CDR_all_seq"]:
    prefix = region.replace("_seq", "")
    desc = descriptor_df[region].apply(seq_desc)
    desc.columns = [f"{prefix}_{c}" for c in desc.columns]
    descriptor_df = pd.concat([descriptor_df, desc], axis=1)
    descriptor_df[f"{prefix}_flags"] = descriptor_df[region].apply(flags)
descriptor_cols = ["Nanobody_id", "Cluster", "Round","Pred_prob_1","Sequence", "Aligned_sequence", "CDR1_aligned", "CDR2_aligned", "CDR3_aligned", "CDR1_seq", "CDR2_seq","CDR3_seq",
    "CDR3_length","CDR3_net_charge","CDR3_positive_count","CDR3_negative_count","CDR3_charged_count","CDR3_aromatic_count","CDR3_hydrophobic_count","CDR3_polar_count",
    "CDR3_cysteine_count","CDR3_glyco_motif","CDR3_max_repeat","CDR3_charge_density","CDR3_aromatic_fraction","CDR3_hydrophobic_fraction","CDR3_polar_fraction","CDR3_flags",
    "CDR_all_length","CDR_all_net_charge","CDR_all_positive_count","CDR_all_negative_count","CDR_all_aromatic_count","CDR_all_hydrophobic_count","CDR_all_polar_count",
    "CDR_all_cysteine_count","CDR_all_glyco_motif","CDR_all_flags",]

descriptor_cols = [c for c in descriptor_cols if c in descriptor_df.columns]
descriptor_table = (
    descriptor_df[descriptor_cols]
    .sort_values(["Cluster", "Pred_prob_1"], ascending=[True, False])
    .copy())
for col in ["Pred_prob_1", "CDR3_charge_density", "CDR3_aromatic_fraction", "CDR3_hydrophobic_fraction", "CDR3_polar_fraction",]:
    if col in descriptor_table.columns:
        descriptor_table[col] = descriptor_table[col].round(3)
descriptor_table.to_csv(descriptor_csv, index=False)
print("Saved descriptor CSV to:", descriptor_csv)
print(f"Best k: {best_k}")
print(f"Best silhouette score: {best_score:.4f}")

if len(X_cluster) >= 3:
    reducer = umap.UMAP(
        n_neighbors=min(umap_n_neighbors, max(2, len(X_cluster) - 1)),
        min_dist=umap_min_dist,
        random_state=random_state,)
    X_umap = reducer.fit_transform(X_cluster)
else:
    X_umap = np.zeros((len(X_cluster), 2), dtype=float)
    if len(X_cluster) >= 1:
        X_umap[:, 0] = np.arange(len(X_cluster))

top_pos["umap_1"] = X_umap[:, 0]
top_pos["umap_2"] = X_umap[:, 1]
unique_labels = sorted(np.unique(labels))
n_clusters = len(unique_labels)
print("KMeans cluster labels:", unique_labels)
print("Number of clusters:", n_clusters)

if aggregation == "mean":
    agg_func = np.mean
elif aggregation == "sum":
    agg_func = np.sum
else:
    raise ValueError("aggregation must be 'mean' or 'sum'")
cluster_heatmaps = {}
cluster_sizes = {}

for cluster_id in unique_labels:
    cluster_indices = np.where(labels == cluster_id)[0]
    cluster_mats = [mats[i] for i in cluster_indices]
    cluster_sizes[cluster_id] = len(cluster_mats)
    cluster_heatmaps[cluster_id] = agg_func(np.stack(cluster_mats, axis=0), axis=0)

base_colors = plt.cm.get_cmap(cluster_map_cmap_name, max(n_clusters, 1))
cluster_id_to_color_index = {cid: i for i, cid in enumerate(unique_labels)}
point_colors = [base_colors(cluster_id_to_color_index[lab] % max(n_clusters, 1)) for lab in labels]
print(len(sequences[0]))
print(len(anarci_positions))
assert len(anarci_positions) == len(sequences[0])

with PdfPages(output_pdf) as pdf:
    #Page 1
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(k_values, silhouette_scores, marker="o")
    axes[0].axvline(best_k, linestyle="--")
    axes[0].set_xlabel("Number of clusters (k)")
    axes[0].set_ylabel("Silhouette score")
    axes[0].set_title("Silhouette search for best k")
    axes[0].grid(alpha=0.25)
    axes[1].scatter(X_umap[:, 0], X_umap[:, 1], c=point_colors, s=35)
    axes[1].set_xlabel("UMAP 1")
    axes[1].set_ylabel("UMAP 2")
    axes[1].set_title("Cosine-KMeans clusters (UMAP only for visualization)")
    sc = axes[2].scatter(X_umap[:, 0], X_umap[:, 1], c=top_pos["Pred_prob_1"].values, cmap="viridis", s=35)
    axes[2].set_xlabel("UMAP 1")
    axes[2].set_ylabel("UMAP 2")
    axes[2].set_title("Pred_prob_1 on UMAP")
    fig.colorbar(sc, ax=axes[2], label="Pred_prob_1")
    fig.suptitle(f"Charged-only cosine-KMeans overview | best_k={best_k} | silhouette={best_score:.4f}", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(10, 5))
    cluster_ids_sorted = sorted(cluster_sizes.keys())
    cluster_counts_sorted = [cluster_sizes[c] for c in cluster_ids_sorted]
    ax.bar([str(c) for c in cluster_ids_sorted], cluster_counts_sorted)
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Number of samples")
    ax.set_title("KMeans cluster sizes")
    ax.grid(axis="y", alpha=0.25)
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    for cluster_id in sorted(cluster_heatmaps.keys()):
        heatmap = cluster_heatmaps[cluster_id]
        n_cluster = cluster_sizes[cluster_id]
        cluster_df = select_top_nonredundant_cluster_examples(
            top_pos[top_pos["Cluster"] == cluster_id],
            n_examples=n_examples_per_cluster)
        if len(cluster_df) == 0:
            continue

        selected_indices = cluster_df.index.to_list()
        selected_sequences = [str(top_pos.loc[idx, "CDRs"]) for idx in selected_indices]
        selected_pred_probs = [float(top_pos.loc[idx, "Pred_prob_1"]) for idx in selected_indices]
        overlap_strength = build_position_overlap_strength(selected_sequences, min_overlap=5)
        heatmap_with_gaps = heatmap
        positions_with_gaps = anarci_positions
        fig = plt.figure(figsize=(18, 8.5))
        gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[1.1, 1.0], hspace=0.35)
        ax_heat = fig.add_subplot(gs[0, 0])
        heatmap_plot = heatmap
        im = ax_heat.imshow(heatmap_plot, aspect="auto", cmap=heatmap_cmap)
        ax_heat.set_xlim(-0.5, heatmap_plot.shape[1] - 0.5)
        ax_heat.set_ylim(heatmap_plot.shape[0] - 0.5, -0.5)
        fig.colorbar(im, ax=ax_heat, label=f"{aggregation} attribution")
        ax_heat.set_xticks(np.arange(len(aa_labels)))
        ax_heat.set_xticklabels(aa_labels, rotation=0, fontsize=9)
        key_positions = ["27", "30", "36", "56", "60", "63", "105", "110", "115", "117"]
        yticks = []
        yticklabels = []
        for pos in key_positions:
            if pos in anarci_positions:
                yticks.append(anarci_positions.index(pos))
                yticklabels.append(pos)
        ax_heat.set_yticks(yticks)
        ax_heat.set_yticklabels(yticklabels, fontsize=10, rotation=30, ha="right", va="center")
        numeric_positions = []
        for p in anarci_positions:
            m = re.match(r"(\d+)", str(p))
            numeric_positions.append(int(m.group(1)) if m else None)
        cdr1_idx = [i for i, p in enumerate(numeric_positions) if p is not None and 27 <= p <= 38]
        cdr2_idx = [i for i, p in enumerate(numeric_positions) if p is not None and 56 <= p <= 65]
        cdr3_idx = [i for i, p in enumerate(numeric_positions) if p is not None and 105 <= p <= 117]
        if cdr1_idx:
            ax_heat.hlines(max(cdr1_idx) + 0.5, -0.5, heatmap_plot.shape[1] - 0.5,
                   colors="white", linewidth=2.0)
        if cdr2_idx:
            ax_heat.hlines(max(cdr2_idx) + 0.5, -0.5, heatmap_plot.shape[1] - 0.5,
                   colors="white", linewidth=2.0)
        x_left = -1.8
        if cdr1_idx:
            ax_heat.text(
                x_left, np.mean(cdr1_idx), "CDR1",
                rotation=90, va="center", ha="center",
                fontsize=11, fontweight="bold")
        if cdr2_idx:
            ax_heat.text(
                x_left, np.mean(cdr2_idx), "CDR2",
                rotation=90, va="center", ha="center",
                fontsize=11, fontweight="bold")
        if cdr3_idx:
            ax_heat.text(
                x_left, np.mean(cdr3_idx), "CDR3",
                rotation=90, va="center", ha="center",
                fontsize=11, fontweight="bold")
        ax_heat.set_xlabel("Amino acid", fontsize=11)
        ax_heat.set_ylabel("ANARCII / IMGT position", fontsize=11)
        round_counts = cluster_df["Round"].astype(str).value_counts().to_dict()
        ax_heat.set_title(
            f"Cluster {cluster_id} aggregated explanation matrix "
            f"(n={n_cluster}) | "
            f"R1:{round_counts.get('1', 0)} "
            f"R2:{round_counts.get('2', 0)} "
            f"R3:{round_counts.get('3', 0)}",
            fontsize=12)
        if len(selected_sequences) > 0:
            ax_seq = fig.add_subplot(gs[1, 0])
            selected_rounds = [top_pos.loc[idx, "Round"] for idx in selected_indices]
            draw_sequence_stack(ax_seq, selected_sequences, selected_pred_probs, selected_rounds, overlap_strength,
                title=(
                    f"Cluster {cluster_id}: top {len(selected_sequences)} non-redundant full sequences by Pred_prob_1\n"
                    "Green background marks positions where at least 5 selected sequences share the same residue"))
            if overlap_strength.size > 0 and np.any(~np.isnan(overlap_strength)):
                overlap_sm = cm.ScalarMappable(
                    cmap=cm.get_cmap(overlap_cmap_name),
                    norm=plt.Normalize(vmin=5, vmax=10))
                overlap_sm.set_array([])
                cbar_overlap = fig.colorbar(overlap_sm, ax=ax_seq, fraction=0.025, pad=0.02)
                cbar_overlap.set_label("Residue overlap count")
            fig.suptitle(f"Cluster {cluster_id}", fontsize=15)
            fig.tight_layout(rect=[0.06, 0, 1, 0.96])
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

            #Page 2
            plots_per_page = 10
            ncols = 2
            nrows = 5
            for page_start in range(0, len(selected_indices), plots_per_page):
                page_indices = selected_indices[page_start:page_start + plots_per_page]
                fig, axes = plt.subplots(nrows, ncols, figsize=(18, 16), squeeze=False)
                for ax, row_idx in zip(axes.flatten(), page_indices):
                    row = top_pos.loc[row_idx]
                    mat = mats[row_idx]
                    seq = str(row["CDRs"])
                    pred_prob = float(row["Pred_prob_1"])
                    nanobody_id = row["Nanobody_id"]
                    pos_attr = []
                    for pos, aa in enumerate(seq):
                        if aa in aa_to_index_full:
                            pos_attr.append(mat[pos, aa_to_index_full[aa]])
                        else:
                            pos_attr.append(0.0)
                    pos_attr = np.array(pos_attr, dtype=float)
                    x = np.arange(len(pos_attr))
                    ax.bar(x, pos_attr)
                    ax.set_xticks(x)
                    ax.set_xticklabels([])
                    for xi, aa in enumerate(seq):
                        ax.text(xi,-0.02,aa,transform=ax.get_xaxis_transform(),ha="center",va="top",fontsize=8)
                    for xi, pos in enumerate(anarci_positions):
                        ax.text(xi,-0.14,str(pos),transform=ax.get_xaxis_transform(), rotation=90, ha="center",va="top",fontsize=6)
                    ax.set_xlabel("Observed residue + ANARCII / IMGT position", labelpad=45)
                    ax.set_ylabel("Observed residue attribution")
                    round_val = row["Round"]
                    ax.set_title(f"{nanobody_id} | R{round_val}\nPred_prob_1={pred_prob:.4f}",
                    fontsize=10)
                    ax.grid(axis="y", alpha=0.25)
                for ax in axes.flatten()[len(page_indices):]:
                    ax.axis("off")
                page_no = page_start // plots_per_page + 1
                fig.suptitle(
                    f"Cluster {cluster_id}: individual attribution profiles "
                    f"(page {page_no})",fontsize=14)
                fig.tight_layout(rect=[0.06, 0.08, 1, 0.96])
                pdf.savefig(fig, bbox_inches="tight")
                plt.close(fig)
print(f"Saved PDF to: {output_pdf}")

Selected top positives: 1981
First positions: ['27', '28', '29', '30', '31', '32', '32A', '33B', '33A', '33']
Last positions: ['112D', '112C', '112B', '112A', '112', '113', '114', '115', '116', '117']
Total positions: 54
Matrix shape: (54, 21)
ANARCII mapping is correct ✅
Sequence length: 54
Matrix rows: 54
ANARCII CDR labels: 54


Building charged-only residue maps: 100%|██████████| 1981/1981 [00:00<00:00, 38839.78it/s]

Charged-only feature matrix shape: (1981, 270)
Normalized clustering matrix shape: (1981, 270)
Searching k in range [5, 40] using silhouette score


k= 5 | silhouette=0.1583
k= 6 | silhouette=0.1641
k= 7 | silhouette=0.1665
k= 8 | silhouette=0.1701
k= 9 | silhouette=0.1749
k=10 | silhouette=0.1745
k=11 | silhouette=0.1726
k=12 | silhouette=0.1785
k=13 | silhouette=0.1829
k=14 | silhouette=0.1718
k=15 | silhouette=0.1780
k=16 | silhouette=0.1478
k=17 | silhouette=0.1304
k=18 | silhouette=0.1331
k=19 | silhouette=0.1397
k=20 | silhouette=0.1356
k=21 | silhouette=0.1431
k=22 | silhouette=0.1321
k=23 | silhouette=0.1392
k=24 | silhouette=0.1428
k=25 | silhouette=0.1315
k=26 | silhouette=0.1347
k=27 | silhouette=0.1264
k=28 | silhouette=0.1426
k=29 | silhouette=0.1289
k=30 | silhouette=0.1366
k=31 | silhouette=0.1340
k=32 | silhouette=0.1335
k=33 | silhouette=0.1248
k=34 | silhouette=0.1338
k=35 | silhouette=0.1262
k=36 | silhouette=0.1304
k=37 | silhouette=0.1342
k=38 | silhouette=0.1264
k=39 | silhouette=0.1303
k=40 | silhouette=0.1317
Selected sequences for descriptor CSV: 260
Saved descriptor CSV to: C:\Users\ankit\Downloads\Lab Tes

c:\Users\ankit\AppData\Local\Programs\Python\Python312\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


KMeans cluster labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Number of clusters: 13
54
54


C:\Users\ankit\AppData\Local\Temp\ipykernel_15924\564661598.py:707: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  base_colors = plt.cm.get_cmap(cluster_map_cmap_name, max(n_clusters, 1))
C:\Users\ankit\AppData\Local\Temp\ipykernel_15924\564661598.py:908: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap=cm.get_cmap(overlap_cmap_name),
C:\Users\ankit\AppData\Local\Temp\ipykernel_15924\564661598.py:916: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0.06, 0, 1, 0.96])
C:\Users\ankit\AppData\Local\Temp\ipykernel_15924\564661598.py:908: MatplotlibDep

Saved PDF to: C:\Users\ankit\Downloads\Lab Test\kmeans_cosine_charged_only_70_test.pdf
